In [2]:
import math
import os

import requests
import pandas as pd
from typing import List
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-US,en;q=0.9",
    "cache-control": "no-cache",
    "pragma": "no-cache",
    "origin": "https://www.adidas.com",
    "referer": "https://www.adidas.com/",
    "sec-ch-ua": '"Chromium";v="127", "Not;A=Brand";v="24", "Google Chrome";v="127"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/127.0.0.0 Safari/537.36"
    ),
}

def create_session():
    """Erstellt eine Session mit Retry-Mechanismus und längeren Timeouts"""
    session = requests.Session()
    
    # Retry-Strategie konfigurieren
    retry_strategy = Retry(
        total=3,  # Maximale Anzahl von Versuchen
        backoff_factor=1,  # Wartezeit zwischen Versuchen (1, 2, 4 Sekunden)
        status_forcelist=[429, 500, 502, 503, 504],  # HTTP-Status-Codes für Retry
        allowed_methods=["GET", "POST"]  # Erlaubte HTTP-Methoden
    )
    
    # Adapter mit Retry-Strategie hinzufügen
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    
    return session


def fetch_reviews(
        model_code: str,
        locale: str = "en_US",
        include: str = "en*",
        limit: int = 10,
        pause: float = 1.0,  # Längere Pause zwischen Anfragen
) -> List[dict]:
    base = f"https://www.adidas.com/api/models/{model_code}/reviews"
    
    session = create_session()
    
    params = {
        "bazaarVoiceLocale": locale,
        "includeLocales": include,
        "sort": "newest",
        "limit": limit,
        "offset": 0
    }

    try:
        print(f"Lade Bewertungen für Modell {model_code}...")
        r = session.get(base, headers=HEADERS, params=params, timeout=(30, 60))  # (connect_timeout, read_timeout)
        r.raise_for_status()
        data = r.json()
        
        total = data["totalResults"]
        pages = math.ceil(total / limit)
        
        print(f"Gefunden: {total} Bewertungen auf {pages} Seiten")
        
        all_rows = []
        for page in range(pages):
            try:
                if page:
                    params["offset"] = page * limit
                    print(f"Lade Seite {page + 1} von {pages}...")
                    
                    time.sleep(pause)
                    
                    r = session.get(base, headers=HEADERS, params=params, timeout=(30, 60))
                    r.raise_for_status()
                    data = r.json()

                for rev in data["reviews"]:
                    if not str(rev.get("locale", "")).lower().startswith("en"):
                        continue
                    
                    # Badges extrahieren - sicherer Ansatz
                    badges = rev.get("badges", [])
                    badge_types = []
                    
                    if isinstance(badges, list):
                        for badge in badges:
                            if isinstance(badge, dict):
                                badge_type = badge.get("badgeType", "")
                                if badge_type:
                                    badge_types.append(badge_type)
                            elif isinstance(badge, str):
                                badge_types.append(badge)
                    
                    # Prüfen ob VerifiedPurchaser oder IncentivizedReview in badges
                    is_verified = "VerifiedPurchaser" in badge_types
                    is_incentivized = "IncentivizedReview" in badge_types
                    
                    all_rows.append(
                        {
                            "id": rev["id"],
                            "date": rev["submissionTime"],
                            "rating": rev["rating"],
                            "title": rev["title"],
                            "text": rev["text"].replace("\n", " "),
                            "nickname": rev["userNickname"],
                            "locale": rev["locale"],
                            "badges": ", ".join(badge_types) if badge_types else "",  # Alle badges als kommagetrennte Liste
                            "isVerifiedPurchaser": is_verified,
                            "isIncentivized": is_incentivized,
                        }
                    )
                    
                    progress = len(all_rows) / total * 100
                    print(f"Review {len(all_rows)} von {total} (Seite {page + 1}/{pages}) - {progress:.1f}%")
                
                print(f"Seite {page + 1} erfolgreich geladen - {len(data['reviews'])} Bewertungen")
                
            except requests.exceptions.Timeout as e:
                print(f"Timeout bei Seite {page + 1}: {e}")
                print("Versuche es erneut...")
                time.sleep(5)  # Längere Pause bei Timeout
                continue
            except requests.exceptions.RequestException as e:
                print(f"Fehler bei Seite {page + 1}: {e}")
                print("Überspringe diese Seite...")
                continue
        
        print(f"Alle Bewertungen erfolgreich geladen: {len(all_rows)} von {total}")
        return all_rows
        
    except requests.exceptions.RequestException as e:
        print(f"Fehler beim Abrufen der Bewertungen: {e}")
        return []
    finally:
        session.close()


def main():
    try:
        print("Starte Adidas Review Scraper...")
        
        # Liste der Model Codes
        model_codes = ['NKT79', 'NKR68', 'NQZ94', 'NMN22', 'LPW32', 'NQW62', 'NJZ74', 'EVOMW1']
        
        # Erstelle data-Ordner falls nicht vorhanden
        os.makedirs("data", exist_ok=True)
        
        # Durchlaufe alle Model Codes
        for i, model_code in enumerate(model_codes, 1):
            print(f"\n{'='*60}")
            print(f"Verarbeite Model {i}/{len(model_codes)}: {model_code}")
            print(f"{'='*60}\n")
            
            data = fetch_reviews(model_code)
            
            if data:
                df = pd.DataFrame(data)
                filename = os.path.join("data", f"adidas_reviews_{model_code}.csv")
                df.to_csv(filename, index=False, encoding='utf-8')
                print(f"✓ Bewertungen erfolgreich in {filename} gespeichert!")
                print(f"✓ Anzahl Bewertungen: {len(data)}")
                
                # Statistik über incentivized reviews
                incentivized_count = df['isIncentivized'].sum()
                verified_count = df['isVerifiedPurchaser'].sum()
                print(f"✓ Davon incentivized: {incentivized_count}")
                print(f"✓ Davon verified purchaser: {verified_count}")
            else:
                print(f"✗ Keine Bewertungen gefunden für {model_code}")
            
            # Pause zwischen Model Codes (außer beim letzten)
            if i < len(model_codes):
                print(f"\nPause von 2 Sekunden vor nächstem Model...")
                time.sleep(2)
        
        print(f"\n{'='*60}")
        print(f"✓ FERTIG! Alle {len(model_codes)} Model Codes verarbeitet!")
        print(f"{'='*60}")
            
    except Exception as e:
        print(f"Unerwarteter Fehler: {e}")


if __name__ == "__main__":
    main()

Starte Adidas Review Scraper...

Verarbeite Model 1/8: NKT79

Lade Bewertungen für Modell NKT79...
Gefunden: 2510 Bewertungen auf 251 Seiten
Review 1 von 2510 (Seite 1/251) - 0.0%
Review 2 von 2510 (Seite 1/251) - 0.1%
Review 3 von 2510 (Seite 1/251) - 0.1%
Review 4 von 2510 (Seite 1/251) - 0.2%
Review 5 von 2510 (Seite 1/251) - 0.2%
Review 6 von 2510 (Seite 1/251) - 0.2%
Review 7 von 2510 (Seite 1/251) - 0.3%
Review 8 von 2510 (Seite 1/251) - 0.3%
Review 9 von 2510 (Seite 1/251) - 0.4%
Review 10 von 2510 (Seite 1/251) - 0.4%
Seite 1 erfolgreich geladen - 10 Bewertungen
Lade Seite 2 von 251...
Review 11 von 2510 (Seite 2/251) - 0.4%
Review 12 von 2510 (Seite 2/251) - 0.5%
Review 13 von 2510 (Seite 2/251) - 0.5%
Review 14 von 2510 (Seite 2/251) - 0.6%
Review 15 von 2510 (Seite 2/251) - 0.6%
Review 16 von 2510 (Seite 2/251) - 0.6%
Review 17 von 2510 (Seite 2/251) - 0.7%
Review 18 von 2510 (Seite 2/251) - 0.7%
Review 19 von 2510 (Seite 2/251) - 0.8%
Review 20 von 2510 (Seite 2/251) - 0.8%